# Euler Equation with CentPy in 2d

### Import packages

In [2]:
# Install the centpy package
!pip install centpy

In [3]:
# Import numpy and centpy for the solution
import numpy as np
import centpy

In [4]:
# Imports functions from matplotlib and setup for the animation
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML
import time

### Equation

We solve the Euler equations in 2D

\begin{equation}
\partial_t
\begin{bmatrix} \rho \\ \rho u_x \\ \rho u_y \\ E \end{bmatrix}
+
\partial_x
\begin{bmatrix} \rho u_x \\ \rho u_x^2 + p \\  \rho u_x u_y \\ (E+p) u_x \end{bmatrix}
+
\partial_y
\begin{bmatrix} \rho u_y \\ \rho u_y u_x \\  \rho u_y^2 +p \\ (E+p) u_y \end{bmatrix}
= 0
\end{equation}

with the equation of state

\begin{equation}
p = (\gamma-1) \left(E-\frac{1}{2} \rho (u_x^2 - u_y^2) \right), \qquad \gamma=1.4
\end{equation}

on the domain $(x,y,t)\in([0,1]\times[0,1]\times[0,0.1])$ with initial data for a *2D Riemann problem*:

\begin{equation}
(\rho, v, p)_{t=0} =
\begin{cases}
(1,0,1) & \text{if} & 0<x\leq0.5 \\
(0.125, 0, 0.1) & \text{if} & 0.5<x<1
\end{cases}
\end{equation}

and Dirichlet boundary data set by initial data on each boundary. The solution is computed using a 200 $\times$ 200 mesh and CFL number 0.75.

In [5]:
pars = centpy.Pars2d(
    x_init=0., x_final=1.,
    y_init=0., y_final=1.,
    J=200, K=200,
    t_final=0.4,
    dt_out=0.005,
    cfl=0.475,
    scheme="sd2",)
pars.gamma = 1.4

In [6]:
# Euler equation for sd2
class Euler2d(centpy.Equation2d):

    # Вспомогательные функции
    def pressure(self, u):
        # p = (gamma - 1) * (E - 0.5 * rho * (u^2 + v^2))
        return (self.gamma - 1.0) * (
            u[..., 3] - 0.5 * (u[..., 1]**2 + u[..., 2]**2) / u[..., 0]
        )

    def euler_data(self):
        gamma = self.gamma
        pone = 1.5; ptwo = 0.3; pthree = 0.029; pfour = 0.3

        upperright, upperleft, lowerright, lowerleft = np.ones((4, 4))

        upperright[0] = 1.5
        upperright[1] = 0.0
        upperright[2] = 0.0
        upperright[3] = pone / (gamma - 1.0) + 0.5 * (upperright[1]**2 + upperright[2]**2) / upperright[0]

        upperleft[0] = 0.5323
        upperleft[1] = 1.206 * upperleft[0]
        upperleft[2] = 0.0
        upperleft[3] = ptwo / (gamma - 1.0) + 0.5 * (upperleft[1]**2 + upperleft[2]**2) / upperleft[0]

        lowerright[0] = 0.5323
        lowerright[1] = 0.0
        lowerright[2] = 1.206 * lowerright[0]
        lowerright[3] = pfour / (gamma - 1.0) + 0.5 * (lowerright[1]**2 + lowerright[2]**2) / lowerright[0]

        lowerleft[0] = 0.138
        lowerleft[1] = 1.206 * lowerleft[0]
        lowerleft[2] = 1.206 * lowerleft[0]
        lowerleft[3] = pthree / (gamma - 1.0) + 0.5 * (lowerleft[1]**2 + lowerleft[2]**2) / lowerleft[0]

        return upperright, upperleft, lowerright, lowerleft

    def initial_data(self):
        u = np.zeros((self.J + 4, self.K + 4, 4))
        upperright, upperleft, lowerright, lowerleft = self.euler_data()

        # Индексы центра пересечения (x=0.5, y=0.5) с учетом ghost-ячеек
        midJ = self.J // 2 + 2
        midK = self.K // 2 + 2

        u[midJ:, midK:] = upperright
        u[:midJ, midK:] = upperleft
        u[midJ:, :midK] = lowerright
        u[:midJ, :midK] = lowerleft

        return u

    def boundary_conditions(self, u):
        upperright, upperleft, lowerright, lowerleft = self.euler_data()

        midJ = self.J // 2 + 2
        midK = self.K // 2 + 2

        # Левая граница (2 слоя ghost-ячеек: x=0)
        u[0:2, midK:] = upperleft
        u[0:2, :midK] = lowerleft

        # Правая граница (2 слоя ghost-ячеек: x=1)
        u[-2:, midK:] = upperright
        u[-2:, :midK] = lowerright

        # Нижняя граница (2 слоя ghost-ячеек: y=0)
        u[:midJ, 0:2] = lowerleft
        u[midJ:, 0:2] = lowerright

        # Верхняя граница (2 слоя ghost-ячеек: y=1)
        u[:midJ, -2:] = upperleft
        u[midJ:, -2:] = upperright

        return u

    def flux_x(self, u):
        f = np.empty_like(u)
        p = self.pressure(u)

        f[..., 0] = u[..., 1]
        f[..., 1] = u[..., 1]**2 / u[..., 0] + p
        f[..., 2] = u[..., 1] * u[..., 2] / u[..., 0]
        f[..., 3] = (u[..., 3] + p) * u[..., 1] / u[..., 0]
        return f

    def flux_y(self, u):
        g = np.empty_like(u)
        p = self.pressure(u)

        g[..., 0] = u[..., 2]
        g[..., 1] = u[..., 1] * u[..., 2] / u[..., 0]
        g[..., 2] = u[..., 2]**2 / u[..., 0] + p
        g[..., 3] = (u[..., 3] + p) * u[..., 2] / u[..., 0]
        return g

    def spectral_radius_x(self, u):
        # Вычисляем радиус только для внутренних ячеек, если передается весь массив с ghost-ячейками
        if u.shape[0] == self.J + 4:
            j0 = slice(2, -2)
            ucore = u[j0, j0]
        else:
            ucore = u

        rho = ucore[..., 0]
        vx = ucore[..., 1] / rho
        vy = ucore[..., 2] / rho
        p = (self.gamma - 1.0) * (ucore[..., 3] - 0.5 * rho * (vx**2 + vy**2))

        # Защита от отрицательных давлений и плотности для стабильности
        p = np.maximum(p, 1e-10)
        rho = np.maximum(rho, 1e-10)

        c = np.sqrt(self.gamma * p / rho)
        return np.abs(vx) + c

    def spectral_radius_y(self, u):
        if u.shape[0] == self.J + 4:
            j0 = slice(2, -2)
            ucore = u[j0, j0]
        else:
            ucore = u

        rho = ucore[..., 0]
        vx = ucore[..., 1] / rho
        vy = ucore[..., 2] / rho
        p = (self.gamma - 1.0) * (ucore[..., 3] - 0.5 * rho * (vx**2 + vy**2))

        p = np.maximum(p, 1e-10)
        rho = np.maximum(rho, 1e-10)

        c = np.sqrt(self.gamma * p / rho)
        return np.abs(vy) + c

In [ ]:
class EulerSod2d(centpy.Equation2d):

    def _compute_pressure(self, q):
        rho, rhou, rhov, E = q[..., 0], q[..., 1], q[..., 2], q[..., 3]
        # Защита от деления на ноль (аналогично JAX версии)
        rho_safe = np.maximum(rho, 1e-10)
        u = rhou / rho_safe
        v = rhov / rho_safe
        return (self.gamma - 1.0) * (E - 0.5 * rho * (u**2 + v**2))

    def flux_x(self, q):
        rho, rhou, rhov, E = q[..., 0], q[..., 1], q[..., 2], q[..., 3]
        rho_safe = np.maximum(rho, 1e-10)
        u = rhou / rho_safe
        p = self._compute_pressure(q)

        f = np.empty_like(q)
        f[..., 0] = rhou
        f[..., 1] = rhou * u + p
        f[..., 2] = rhou * (rhov / rho_safe)
        f[..., 3] = u * (E + p)
        return f

    def flux_y(self, q):
        rho, rhou, rhov, E = q[..., 0], q[..., 1], q[..., 2], q[..., 3]
        rho_safe = np.maximum(rho, 1e-10)
        v = rhov / rho_safe
        p = self._compute_pressure(q)

        g = np.empty_like(q)
        g[..., 0] = rhov
        g[..., 1] = rhou * v
        g[..., 2] = rhov * v + p
        g[..., 3] = v * (E + p)
        return g

    def spectral_radius_x(self, q):
        rho, rhou = q[..., 0], q[..., 1]
        rho_safe = np.maximum(rho, 1e-10)
        u = rhou / rho_safe
        p = np.maximum(self._compute_pressure(q), 1e-10)
        return np.abs(u) + np.sqrt(self.gamma * p / rho_safe)

    def spectral_radius_y(self, q):
        rho, rhov = q[..., 0], q[..., 2]
        rho_safe = np.maximum(rho, 1e-10)
        v = rhov / rho_safe
        p = np.maximum(self._compute_pressure(q), 1e-10)
        return np.abs(v) + np.sqrt(self.gamma * p / rho_safe)

    def initial_data(self):
        x = self.x

        # 1D задача Сода: разрыв только по оси X
        rho = np.where(x < 0.5, 1.0, 0.125)
        vx = np.zeros_like(x)
        vy = np.zeros_like(x)
        p = np.where(x < 0.5, 1.0, 0.1)

        E = p / (self.gamma - 1.0) + 0.5 * rho * (vx**2 + vy**2)

        u = np.empty((self.J + 4, self.K + 4, 4))
        u[..., 0] = rho
        u[..., 1] = rho * vx
        u[..., 2] = rho * vy
        u[..., 3] = E
        return u

    def boundary_conditions(self, u):
        # Экстраполяция Неймана (нулевой градиент)
        # Копируем крайние внутренние ячейки (индексы 2 и -3) в ghost-зоны
        u[0, :] = u[2, :]
        u[1, :] = u[2, :]
        u[-1, :] = u[-3, :]
        u[-2, :] = u[-3, :]

        u[:, 0] = u[:, 2]
        u[:, 1] = u[:, 2]
        u[:, -1] = u[:, -3]
        u[:, -2] = u[:, -3]
        return u

### Solution

In [1]:
eqn = Euler2d(pars)
t0 = time.time()
soln = centpy.Solver2d(eqn)
soln.solve()
t1 = time.time()
print(f"\n[СPU centpy] Чистое время выполнения: {t1 - t0:.4f} секунд")

NameError: name 'Euler2d' is not defined

In [8]:
def run_centpy_with_different_mesh(grids: list[int]):
    """
    Запускает CPU-решатель centpy последовательно на массиве различных сеток.

    :param grids: Список размеров сеток (например, [50, 100, 200])
    :return: Словарь с результатами и временем выполнения для каждой сетки
    """
    all_results = {}

    for N in grids:
        print(f"\n{'='*50}")
        print(f"Инициализация сетки centpy: {N}x{N}")
        print(f"{'='*50}")

        # 1. Создаем параметры для текущей сетки (J - по X, K - по Y)
        pars = centpy.Pars2d(
            x_init=0., x_final=1.,
            y_init=0., y_final=1.,
            J=N, K=N,
            t_final=0.4,
            dt_out=0.005,
            cfl=0.475,
            scheme="sd2"
        )
        pars.gamma = 1.4

        # 2. Инициализируем уравнение и решатель
        eqn = Euler2d(pars)
        soln = centpy.Solver2d(eqn)

        # 3. Основной запуск и бенчмарк (без прогрева)
        print(f"--- Запуск CPU Бенчмарка (centpy Euler 2D) ---")
        t0 = time.time()

        soln.solve()

        t1 = time.time()
        elapsed_time = t1 - t0

        print(f"[CPU centpy] Чистое время выполнения для {N}x{N}: {elapsed_time:.4f} секунд")

        # 4. Сохраняем результаты
        all_results[N] = {
            'time': elapsed_time,
            'soln': soln,         # Сохраняем весь объект решателя, если нужны внутренние поля
            'u_final': soln.u   # В centpy обычно soln.u хранит текущее состояние решения
        }

    return all_results


if __name__ == "__main__":
    # Задаем массив сеток
    grid_sizes = [40, 80, 160, 200, 320]

    # Запускаем цикл для centpy
    centpy_results = run_centpy_with_different_mesh(grid_sizes)

    # Сравнение времени (если вы объедините JAX и centpy скрипты):
    # for N in grid_sizes:
    #     print(f"Сетка {N}x{N}: JAX = {jax_results[N]['time']:.4f} с | centpy = {centpy_results[N]['time']:.4f} с")



Инициализация сетки centpy: 40x40
--- Запуск CPU Бенчмарка (centpy Euler 2D) ---
[CPU centpy] Чистое время выполнения для 40x40: 0.8158 секунд

Инициализация сетки centpy: 80x80
--- Запуск CPU Бенчмарка (centpy Euler 2D) ---
[CPU centpy] Чистое время выполнения для 80x80: 2.0834 секунд

Инициализация сетки centpy: 160x160
--- Запуск CPU Бенчмарка (centpy Euler 2D) ---
[CPU centpy] Чистое время выполнения для 160x160: 16.6495 секунд

Инициализация сетки centpy: 200x200
--- Запуск CPU Бенчмарка (centpy Euler 2D) ---
[CPU centpy] Чистое время выполнения для 200x200: 29.8197 секунд

Инициализация сетки centpy: 320x320
--- Запуск CPU Бенчмарка (centpy Euler 2D) ---
[CPU centpy] Чистое время выполнения для 320x320: 130.5105 секунд


In [ ]:
from re import X
import numpy as np
from google.colab import files

np.savez('centpy_data.npz',
         X=soln.x,
         Y=soln.y,
         u=soln.u)

# Команда для автоматического скачивания файла из Colab
files.download('centpy_data.npz')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Animation

In [ ]:
# Animation
fig, ax = plt.subplots()
ax.set_xlim(soln.x_init, soln.x_final)
ax.set_ylim(soln.y_init, soln.y_final)

# Извлекаем сетку координат и начальные данные для первого кадра
x_grid = soln.x[1:-1]
y_grid = soln.y[1:-1]
data_init = soln.u_n[0, 1:-1, 1:-1, 0]

# 1. Создаем фоновую тепловую карту с помощью imshow
# Параметр extent привязывает матрицу к реальным координатам
im = ax.imshow(
    data_init,
    extent=[soln.x_init, soln.x_final, soln.y_init, soln.y_final],
    origin='lower',             # Гарантирует, что ось Y направлена вверх
    cmap='coolwarm',             # Или 'magma', 'viridis', 'turbo'
    interpolation='bicubic',    # Сглаживание для профессионального вида
    aspect='auto'               # Позволяет осям масштабироваться независимо
)

# Добавляем цветовую шкалу для наглядности (опционально, но рекомендуется)
cbar = fig.colorbar(im, ax=ax)
cbar.set_label('Значение u_n')

# 2. Отрисовываем начальные контуры поверх тепловой карты
ax.contour(
    x_grid, y_grid, data_init,
    levels=20,
    colors='black',
    alpha=0.5,
    linewidths=0.5
)

# Функция обновления для каждого кадра анимации
def animate(i):
    # Получаем данные текущего шага
    data = soln.u_n[i, 1:-1, 1:-1, 0]

    # Быстрое обновление тепловой карты (работает быстрее, чем перерисовка)
    im.set_data(data)

    # Если глобальный минимум и максимум меняются со временем,
    # можно раскомментировать следующую строку для динамической шкалы:
    # im.set_clim(vmin=data.min(), vmax=data.max())

    # Удаляем старые линии контуров из коллекции осей
    for c in ax.collections:
        c.remove()

    # Рисуем новые контурные линии для текущего кадра
    ax.contour(
        x_grid, y_grid, data,
        levels=20,
        colors='black',
        alpha=0.5,
        linewidths=0.5
    )

    return [im]

plt.close() # Закрываем статичную фигуру, чтобы она не дублировалась в выводе

# Создаем анимацию
anim = animation.FuncAnimation(fig, animate, frames=soln.Nt, interval=100, blit=False)

# Выводим как HTML5 видео
HTML(anim.to_html5_video())